# 07 — 전체 활성 노드 감사와 실패 원인 확인

## 범위와 성공 기준
2026-10-19 최초 승인: **기존 Before 14개 분석 + GPU 없는 재현**. 이후 전체 활성 노드 감사와 2026-10-20 유료 수집을 승인받아 14개 실제 기록 수집·백업·Pod 중지를 완료했다. 최신 실제 결과는 **8절부터** 읽는다. 서비스 수정·노드 재설계·Git 반영은 별도 승인 대상이다.

1. 저장된 질문/검색/생성 입력/답변과 사용자 의견을 연결한다 → 14개 모두 관찰·원인 후보·추가 확인을 구분한다.
2. 실제 서비스 함수와 그래프의 외부 모델/검색 경계만 모의 응답으로 바꾼다 → 이력 전달·문맥·인용·분기·실패·빈 응답·취소·느린 소비자를 검증한다.
3. 확인된 동작과 미확정 원인을 분리한다 → 수정 제안과 추가 실행 범위를 기록한다.

기존 점수는 변경하지 않는다. Q01/Q09/Q12의 조건과 Q10의 정책 사실 평가는 보류를 유지한다. 과거 노드 입력은 저장되지 않았다면 복원했다고 주장하지 않는다. 이 노트북의 모의 실행은 성능 평가가 아니다.

## 실제 경로
- 스트리밍: 서비스 → 민감정보 → 질문 이해 → 검색 → 관련성 → 인용 준비 → 그래프 밖 생성 → SSE/인용/종료.
- 비스트리밍: 민감정보 → 질문 이해 → 검색 → 관련성 → 생성 → 인용.
- 민감정보·확답 차단·관련성 실패는 중간에 종료할 수 있다. 과거 formatter/expression_revision 파일의 존재가 현재 실행을 의미하지 않는다.

## 방법과 참고 자료
문제는 정보가 어느 경계에서 사라지는지 모른다는 것이다. 외부 응답을 고정하고 실제 함수의 입력·출력을 비교해 연결 오류와 모델 품질을 분리한다.
- [Python mock 공식 문서](https://docs.python.org/3/library/unittest.mock.html): 외부 모델과 검색만 대체한다. 실제 검색 품질·법률 정확성은 검증하지 못한다.
- [LangGraph 스트리밍 공식 자료](https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/streaming.mdx): updates는 노드 변경, values는 누적 상태를 보여준다. 설치 버전의 기본 API만 쓰며 업그레이드하지 않는다. 전체 debug 로그 대신 아래 모의 그래프에서 변경 키를 확인한다.
- [Python asyncio Queue](https://docs.python.org/3/library/asyncio-queue.html): 꽉 찬 큐에 put_nowait하면 QueueFull이 발생한다. 느린 소비자 실험은 이 조건을 작게 만든다. 해결책 구현은 다음 승인 단계다.

논문 기법이나 reranker를 추가하기 전에 연결 계약부터 확인하는 것이 이번 최소 실험이다.

**범위 보완 승인:** 사용자가 모든 활성 노드 점검 및 필요한 실제 기록 수집을 요청했다. 앞부분의 GPU 없는 3A는 부분 완료다. 전체 감사는 아래 노드별 완료/미확인 표를 기준으로 진행한다. 유료 실행 한도는 새로 승인받으며 서비스 동작은 보존한다. PDF 그래프는 이번 채팅 Before 경로에 포함되지 않는다.

읽는 순서: 본문에서 기존·실제 기록과 재현 결과를 확인하고, 실제 재수집이 필요할 때만 마지막 부록 A를 사용한다. 분석 뒤의 짧은 검증 셀은 예상 결과를 확인한다. 실행 승인·시간 제한은 별도의 명시적 오류 검사로 보호한다.

In [1]:
import asyncio
import hashlib
import json
import os
import re
import sys
import threading
from pathlib import Path
from types import ModuleType
from unittest.mock import Mock, patch

import nbformat
import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", None)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
manifest = json.loads((ROOT / "notebooks/redesign/data/before_manifest.json").read_text(encoding="utf-8"))
CACHE = Path(os.environ.get("AIGO_CACHE_DIR", Path.home() / ".cache/aigo-ai"))
CORPUS = CACHE / "before" / manifest["corpus_sha256"]
RUN_ID = "20261018T073552164357Z"
RESULTS = CORPUS / "runs" / RUN_ID
items = json.loads((ROOT / "notebooks/redesign/eval/eval_set.json").read_text(encoding="utf-8"))["items"]
review_path = ROOT / "notebooks/redesign/03_before_review.ipynb"
review = nbformat.read(review_path, as_version=4)
notes = {c.id.split("-")[0].upper(): c.source for c in review.cells if c.id.endswith("-notes")}
answers = {q["id"]: json.loads((RESULTS / f"{q['id']}.json").read_text(encoding="utf-8")) for q in items}
# 원문은 읽기만 하고, 공개 노트북에는 본문 전체 대신 관찰 결과만 남긴다.
protected = [review_path, CORPUS / "corpus.jsonl", *[RESULTS / f"{q['id']}.json" for q in items]]
before_hashes = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
print('[기록][완료] Before 14개·사용자 검토 메모 로드. 저장된 자료만 읽으며 모델을 호출하지 않습니다.')

[기록][완료] Before 14개·사용자 검토 메모 로드. 저장된 자료만 읽으며 모델을 호출하지 않습니다.


In [2]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert len(answers) == len(notes) == 14, '[검증 실패] Before 기록 개수·질문·실행 종류: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(answers[q["id"]]["question"] == q["question"] for q in items), '[검증 실패] Before 기록 개수·질문·실행 종류: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(r["run_kind"] == "reconstructed_before" for r in answers.values()), '[검증 실패] Before 기록 개수·질문·실행 종류: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

## 1. 기존 기록에서 확인할 수 있는 것
검색 결과, 관련성 통과 여부, 검색용 질문, 생성 메시지와 최종 답변은 저장되어 있다. 당시 문서 유형 필터·민감정보/질문 이해의 전체 출력·노드별 시간은 저장되어 있지 않다.

본문 포함 검사는 **문자열 전달 검사**다. 참인 내용인지, 정답 근거를 충분히 찾았는지에 대한 점수가 아니다. 기록의 generation_messages는 당시 빌더로 재구성한 관찰값이며 모델 서버가 실제 수신한 요청 로그는 아니다.

In [3]:
rows = []
for qid, record in answers.items():
    trace = record["trace"]
    docs = trace.get("retrieved_docs") or []
    messages = trace.get("generation_messages") or []
    context = "\n".join(m["content"] for m in messages)
    rows.append({
        "ID": qid, "검색어": trace.get("user_query"), "문서수": len(docs),
        "생성입력기록": bool(messages), "관련성통과": trace.get("retrieval_passed"),
        "본문전달수": sum(bool(d.get("text")) and d["text"] in context for d in docs),
        "인용수": len(record["citations"]),
        "기존정성점수": int(re.search(r"점수: \*\*(\d)/5", notes[qid]).group(1)),
    })
observations = pd.DataFrame(rows)
display(observations)
print('[기록][관찰] Q05 검색어에 고양이 조건이 없습니다. 생성 모델에는 원 질문도 전달되므로 오답의 단독 원인으로 확정하지 않습니다.')

,ID,검색어,문서수,생성입력기록,관련성통과,본문전달수,인용수,기존정성점수
0,Q01,월세 계약 중간 해지 절차,5,True,True,5,5,2
1,Q02,전세 계약 만료 전 특약 전세권 설정으로 인한 집주인 연락 두절 시 계약 파기 가능 여부,0,False,False,0,0,1
2,Q03,계약 해지 사유로 귀신 출현 가능한가,5,True,True,5,5,3
3,Q04,전세자금 대출 승인 후에 임대차 계약 효력 발생 특약 가능 여부,5,True,True,5,5,2
4,Q05,특약 작성 방법,5,True,True,5,2,1
5,Q06,임대차 계약서에서 '차임'의 의미,5,True,True,5,2,3
6,Q07,윗집 소음 해결 방법,5,True,True,5,2,3
7,Q08,공인중개사 준비 방법,5,True,True,5,3,1
8,Q09,계약 무산 시 선지불 보증금 반환 가능 여부,5,True,True,5,1,2
9,Q10,자취생 주거비 지원 정책,5,True,True,5,4,2


[기록][관찰] Q05 검색어에 고양이 조건이 없습니다. 생성 모델에는 원 질문도 전달되므로 오답의 단독 원인으로 확정하지 않습니다.


In [4]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert observations.loc[observations.ID == "Q02", "문서수"].item() == 0, '[검증 실패] Before Q02 검색 개수·Q05 검색어: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert answers["Q05"]["trace"]["user_query"] == "특약 작성 방법", '[검증 실패] Before Q02 검색 개수·Q05 검색어: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

## 2. 질문별 관찰 → 원인 후보 → 다음 확인
아래 오류 설명은 사용자 확정 검토 의견의 요약이다. 새로운 법률 자문·재채점이 아니다. 출처 내용의 정확성은 원 검토 노트북에서 확인한다. 우선순위 Q05/Q08/Q11/T01/T02를 먼저 표시하되 심각한 오류 여부를 새로 추정하지 않는다.

In [5]:
analysis = {
 "Q01": ("최초/갱신을 나누지 않은 중도해지 설명", "조건 누락·근거 적용", "미확정 계약 조건 유지; 검색 본문과 주장 대응 확인"),
 "Q02": ("문서 0개·생성 미실행·일반 추가질문", "검색/필터/코퍼스 중 원인 미확정", "필터 기록이 없어 판별 불가; 코퍼스 보유 확인 후 필터 유무 비교"),
 "Q03": ("해지 판례 5개; 검토 의견은 수선·합의 요건 과장 지적", "생성 해석·사례 적합성", "화재 사건 본문의 조건과 생성 문장 대조"),
 "Q04": ("대출 조건 질문에 판례·임대료 해석례 검색", "직접 근거 누락·생성 문안 혼합", "민법 제147조 보유/검색 여부와 입력 근거 대응 확인"),
 "Q05": ("검색어에서 고양이·부담 최소화 조건 소실", "질문 요약 손실 + 생성 과잉 제안", "원 질문 검색과 비교 필요; 검색어 손실만으로 원인 확정 금지"),
 "Q06": ("정의 근거와 여러 과거 시행령 판본 동시 검색", "불필요한 확장·판본 구별 부족", "정의 문맥만으로 설명하는 비교는 후속 생성 실험"),
 "Q07": ("층간소음 법령 검색; 검토 의견은 적용 범위·항 번호 문제 지적", "생성 해석·주택 조건 미확인", "제20조 본문과 주장/항 번호 대조; 판본의 날짜 판단 별도"),
 "Q08": ("시험 준비 질문에 민법·거래신고·설명의무 판례", "범위 분기·직접 근거 부족·생성", "eval 범위와 실제 라우팅 대조; 외부 도구 미구현 유지"),
 "Q09": ("같은 제3조의2의 판본 5개 검색", "근거 다양성 부족·계약 미성립 가정", "계약 성립 조건 보류; 중복 판본과 다른 조문 보유 확인"),
 "Q10": ("정책 질문에 임대차 법령/판례; 외부 도구 없음", "코퍼스 범위·근거 없는 확장", "정책 사실 보류; 한계 고지·추가 질문만 검토"),
 "Q11": ("민법 제629조 등 검색; 사용자 검토는 결론 뒤집힘 지적", "근거 해석·필수 근거 누락 후보", "제629조 입력 본문과 문제 문장 대조; 제319조 보유 조사"),
 "Q12": ("검색어에서 날짜 소실; 생성에는 원 질문 전달", "조건 확인·기한 표현·판본 적용", "계약 연도 보류; 원 입력에 있던 조건과 답변 대조"),
 "T01": ("검색어에서 2019/2021 조건 소실", "질문 요약·기간 해석·부칙 누락 후보", "기간 조문/부칙 보유 확인; 답변 입력의 날짜 존재 대조"),
 "T02": ("검색어에 2021년 5월 유지, 이후 시행 조문 검색", "판본 적용·별도 신고제 혼동", "시행일/부칙 보유 및 모델 입력 메타데이터 전달 확인"),
}
priority = ["Q05", "Q08", "Q11", "T01", "T02"]
order = priority + [q for q in answers if q not in priority]
display(pd.DataFrame([
    {"ID": q, "관찰/기존의견": analysis[q][0], "원인후보(미확정)": analysis[q][1],
     "추가확인": analysis[q][2], "중간노드기록": "없음"} for q in order
]))

# 보유 여부는 문서 제목·조문 필드로만 확인한다. 본문 속 간접 인용은 세지 않는다.
corpus = [json.loads(line) for line in (CORPUS / "corpus.jsonl").read_text(encoding="utf-8").splitlines()]
targets = [("Q04", "민법", "제147조"), ("Q09", "민법", "제565조"),
           ("Q09", "민법", "제567조"), ("Q11", "형법", "제319조"),
           ("T01", "주택임대차보호법", "제6조")]
coverage = []
for qid, law, article in targets:
    matches = [d for d in corpus if (d.get("law_name") or d.get("title")) == law and d.get("article") == article]
    coverage.append({"ID": qid, "확인대상": f"{law} {article}", "코퍼스판본수": len(matches),
                     "판정범위": "별도 문서 보유 여부; 정답 판본/부칙 충분성은 미검증"})
display(pd.DataFrame(coverage))

,ID,관찰/기존의견,원인후보(미확정),추가확인,중간노드기록
0,Q05,검색어에서 고양이·부담 최소화 조건 소실,질문 요약 손실 + 생성 과잉 제안,원 질문 검색과 비교 필요; 검색어 손실만으로 원인 확정 금지,없음
1,Q08,시험 준비 질문에 민법·거래신고·설명의무 판례,범위 분기·직접 근거 부족·생성,eval 범위와 실제 라우팅 대조; 외부 도구 미구현 유지,없음
2,Q11,민법 제629조 등 검색; 사용자 검토는 결론 뒤집힘 지적,근거 해석·필수 근거 누락 후보,제629조 입력 본문과 문제 문장 대조; 제319조 보유 조사,없음
3,T01,검색어에서 2019/2021 조건 소실,질문 요약·기간 해석·부칙 누락 후보,기간 조문/부칙 보유 확인; 답변 입력의 날짜 존재 대조,없음
4,T02,"검색어에 2021년 5월 유지, 이후 시행 조문 검색",판본 적용·별도 신고제 혼동,시행일/부칙 보유 및 모델 입력 메타데이터 전달 확인,없음
5,Q01,최초/갱신을 나누지 않은 중도해지 설명,조건 누락·근거 적용,미확정 계약 조건 유지; 검색 본문과 주장 대응 확인,없음
6,Q02,문서 0개·생성 미실행·일반 추가질문,검색/필터/코퍼스 중 원인 미확정,필터 기록이 없어 판별 불가; 코퍼스 보유 확인 후 필터 유무 비교,없음
7,Q03,해지 판례 5개; 검토 의견은 수선·합의 요건 과장 지적,생성 해석·사례 적합성,화재 사건 본문의 조건과 생성 문장 대조,없음
8,Q04,대출 조건 질문에 판례·임대료 해석례 검색,직접 근거 누락·생성 문안 혼합,민법 제147조 보유/검색 여부와 입력 근거 대응 확인,없음
9,Q06,정의 근거와 여러 과거 시행령 판본 동시 검색,불필요한 확장·판본 구별 부족,정의 문맥만으로 설명하는 비교는 후속 생성 실험,없음


,ID,확인대상,코퍼스판본수,판정범위
0,Q04,민법 제147조,1,별도 문서 보유 여부; 정답 판본/부칙 충분성은 미검증
1,Q09,민법 제565조,1,별도 문서 보유 여부; 정답 판본/부칙 충분성은 미검증
2,Q09,민법 제567조,1,별도 문서 보유 여부; 정답 판본/부칙 충분성은 미검증
3,Q11,형법 제319조,1,별도 문서 보유 여부; 정답 판본/부칙 충분성은 미검증
4,T01,주택임대차보호법 제6조,5,별도 문서 보유 여부; 정답 판본/부칙 충분성은 미검증


In [6]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert set(analysis) == set(answers), '[검증 실패] 질문별 분석 ID 구성: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

## 3. 모델·검색 응답을 고정해 코드 동작 확인하기
다음은 실제 노드·서비스 함수를 불러온다. 모델과 벡터 저장소 모듈은 import 전에 대체하므로 모델 다운로드나 GPU 연결이 없다. 모의 응답을 정하지 않은 호출은 즉시 실패한다. 별도 커널에서 순서대로 실행한다.

테스트의 `assert`는 현재 결함/동작이 재현됐는지 검사한다. 통과했다고 서비스가 정상이라는 뜻은 아니다. `결함 재현`, `설계 한계 확인`, `정상 경로 확인`을 구분한다.

`[Mock]`은 실제 모델·DB 대신 테스트용 응답을 사용한다는 표시다. 임베딩 계산을 수행했다는 뜻이 아니다. 준비 메시지와 응답 미설정 오류를 구분한다. 모의 모듈을 먼저 등록한 뒤 일반 `import 경로 as 별칭`으로 검사 대상 모듈을 불러온다. 현재 패키지에서 이름 충돌이 없음을 확인했으므로 동적 import는 필요하지 않다.

검증(2026-10-20): 일반 import로 변경 후 새 커널 전체 실행 통과. 검사 대상의 ModuleType 확인과 기존 모의/기록 검증을 통과했다. 비활성 원격 셀은 문법만 확인했으며 GPU에서 재실행하지 않았다.

### 출력 메시지 읽는 법
- `[Mock]`: 고정 응답/합성 입력을 사용하는 검사. `결함 재현`은 문제를 확인했다는 뜻이며 해결됐다는 뜻이 아니다.
- `[기록]`: 저장된 실제 실행 자료를 읽은 결과. `[실행][건너뜀]`은 원격 실행 셀이 비활성 상태라는 뜻이다.
- `[검증][완료]`: 해당 검사의 예상 결과 확인. `[검증 실패]`는 예상 결과와 다르므로 해당 assert의 입력·출력을 확인한다.
- 서비스가 출력하는 원래 `error`/`warning` 로그는 유지한다. 의도적 예외 검사 시작 안내 뒤의 로그는 재현 자료이며, 예상하지 못한 예외는 숨기지 않고 실행을 중단한다.


In [7]:
# 전용 커널에서만 실행: 실제 모델 모듈이 이미 로드됐다면 재시작한다.
if not ('app.llm' not in sys.modules):
    raise ValueError('[Mock][환경 오류] app.llm이 이미 로드되어 격리를 보장할 수 없습니다. 커널을 재시작하고 위에서부터 실행하세요.')
model_module = ModuleType("app.llm")
model_module.llm = Mock()
model_module.streaming_llm = Mock()
model_module.llm.invoke.side_effect = AssertionError("[Mock] LLM 응답 미설정: 테스트 응답을 지정하세요.")
model_module.streaming_llm.stream.side_effect = AssertionError("[Mock] 스트리밍 응답 미설정: 테스트 청크를 지정하세요.")
vector_module = ModuleType("app.vectordb")
vector_module.Embedder = Mock(side_effect=AssertionError("[Mock] 임베더 생성 요청 감지: 이 검사는 고정 검색 결과를 사용합니다."))
vector_module.QdrantStore = Mock(side_effect=AssertionError("[Mock] 벡터 저장소 생성 요청 감지: 검색 응답을 Mock으로 지정하세요."))
# app.__init__가 무관한 PDF/API까지 가져오므로 패키지 경로만 제공한다.
app_package = ModuleType("app")
app_package.__path__ = [str(ROOT / "app")]
sys.modules.update({"app": app_package, "app.llm": model_module, "app.vectordb": vector_module})
import app.graph.pipeline as pipeline
import app.services.chat_service as chat
import app.graph.nodes.generator as gen
import app.graph.nodes.retrieve as retrieval
import app.graph.nodes.check_relevance as relevance
import app.graph.nodes.resolve_citations as citations
import app.graph.nodes.understand_query as understand
# 함수가 아닌 모듈 객체를 가져왔는지 확인한다.
assert all(isinstance(module, ModuleType) for module in
           [pipeline, chat, gen, retrieval, relevance, citations, understand]), '[검증 실패] 일반 import의 모듈 객체: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
print("[Mock] LLM·스트리밍 응답 대체 준비 완료")
print("[Mock] 임베더·벡터 저장소 격리 완료: 실제 임베딩 계산·DB 연결 없음")
from langchain_core.messages import AIMessage, HumanMessage
from importlib.metadata import version
print("[환경][정보] 로컬 검증 패키지:", {name: version(name) for name in ["langgraph", "langchain-core", "nbclient"]})
checks = []

def record(name, kind, detail):
    """재현 이름·분류·관찰을 결과 목록에 추가한다. 점수를 계산하지 않는다."""
    checks.append({"검사": name, "분류": kind, "관찰": detail})
    print(f"[Mock][{kind}] {name}: {detail}")

async def consume(stream):
    """유한한 모의 SSE 스트림을 받아 이벤트 목록으로 반환한다."""
    return [event async for event in stream]

[Mock] LLM·스트리밍 응답 대체 준비 완료
[Mock] 임베더·벡터 저장소 격리 완료: 실제 임베딩 계산·DB 연결 없음
[환경][정보] 로컬 검증 패키지: {'langgraph': '1.1.6', 'langchain-core': '1.2.27', 'nbclient': '0.11.0'}


### 3-1. 대화 이력·계약 정보와 생성 메시지
예: 앞 대화에 계약 날짜가 있는데 후속 질문에는 날짜를 다시 쓰지 않았다. 서비스에 넘긴 정보가 생성 경계까지 가는지 확인한다. 현재 Before 14개는 독립 질문이므로 이 결함을 기존 오답의 원인으로 연결하지 않는다.

In [8]:
seen = {}
def fake_preformat(query):
    """실제 서비스가 넘긴 입력을 기록하고 최소 상태를 반환한다."""
    seen["query"] = query
    return {"user_input": query, "retrieved_docs": []}

def fake_formatter(state):
    """생성 직전 상태를 기록하고 고정 문자열을 반환한다."""
    seen["state"] = dict(state)
    yield "모의 답변"

with patch.object(chat, "run_preformat", fake_preformat), patch.object(chat, "stream_formatter", fake_formatter):
    events = await consume(chat.stream_chat("언제 끝나나요?", history=[{"role":"user", "content":"계약일은 2020년 1월입니다"}], contract_context={"has_contract": True}))
assert "messages" not in seen["state"] and "contract_context" not in seen["state"], '[검증 실패] 이력·생성 답변·문맥 메타데이터 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("이력/계약 정보 전달", "결함 재현", "서비스가 받은 history와 contract_context가 다음 경계에 없음")

state = {"user_input": "질문", "messages": [HumanMessage(content="이전 질문"), AIMessage(content="이전 답변")], "retrieved_docs": []}
assert len(gen._build_messages(state)) == 4, '[검증 실패] 이력·생성 답변·문맥 메타데이터 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'  # 빌더 자체는 전달받은 이력을 사용한다.
with patch.object(gen.llm, "invoke", return_value=AIMessage(content="새 답변"), side_effect=None):
    update = gen.generator(state)
assert update["final_answer"] == "새 답변" and all(m.type != "ai" for m in update["messages"]), '[검증 실패] 이력·생성 답변·문맥 메타데이터 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("생성 답변의 이력 저장", "결함 재현", "final_answer는 반환하지만 messages 갱신에는 사용자 질문만 있음")

fixture = {"title":"모의법", "doc_type":"법령", "text":"모의 본문", "source_id":"fixture", "조문번호":"제1조", "version":"판본표식", "effective_from":"2020-01-01"}
prompt = "\n".join(m.content for m in gen._build_messages({"user_input":"질문", "retrieved_docs":[fixture]}))
assert "모의 본문" in prompt and "판본표식" not in prompt and "2020-01-01" not in prompt, '[검증 실패] 이력·생성 답변·문맥 메타데이터 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("문맥 메타데이터", "설계 한계 확인", "본문·제목·유형 전달, 별도 version/effective_from 필드는 미전달; 본문 내 날짜까지 없다는 뜻은 아님")

[Mock][결함 재현] 이력/계약 정보 전달: 서비스가 받은 history와 contract_context가 다음 경계에 없음
[Mock][결함 재현] 생성 답변의 이력 저장: final_answer는 반환하지만 messages 갱신에는 사용자 질문만 있음
[Mock][설계 한계 확인] 문맥 메타데이터: 본문·제목·유형 전달, 별도 version/effective_from 필드는 미전달; 본문 내 날짜까지 없다는 뜻은 아님


### 3-2. 실제 그래프의 분기와 노드 변경
검색기 응답과 두 모델 응답만 고정한다. 그래프 자체는 실제 build_graph를 사용한다. 인용과 생성의 순서가 다른지, 문서가 비어 있으면 생성이 생략되는지 확인한다.

In [9]:
parsed = json.dumps({"intent_metadata": {"doc_type": "법령"}, "is_definitive": False, "user_query": "모의 검색어"})
doc = {**fixture, "score": 1.0}
store = Mock()
store.search.return_value = [doc]
paths = {}
for include in [False, True]:
    replies = [AIMessage(content=parsed)] + ([AIMessage(content="모의 생성 답변")] if include else [])
    with patch.object(retrieval, "_get_store", return_value=store), patch.object(gen.llm, "invoke", side_effect=replies):
        graph = pipeline.build_graph(include_generator=include)
        events = list(graph.stream({"user_input": "모의 질문"}, stream_mode=["updates", "values"]))
        updates = [value for mode, value in events if mode == "updates"]
    paths[include] = [name for step in updates for name in step]
    # 선형 경로에서 직전 values가 다음 노드의 입력 상태다. 변경분과 누적 상태를 구분한다.
    previous = {}
    boundaries = []
    for mode, value in events:
        if mode == "values":
            previous = value
        else:
            for name, delta in value.items():
                boundaries.append({"노드": name, "입력질문": previous.get("user_input"),
                    "입력검색어": previous.get("user_query"),
                    "입력문서수": len(previous.get("retrieved_docs") or []),
                    "입력이력수": len(previous.get("messages") or []),
                    "출력변경키": ", ".join(delta),
                    "출력검색통과": delta.get("retrieval_passed")})
    display(pd.DataFrame(boundaries))
assert paths[False][-1] == "resolve_citations" and "generator" not in paths[False], '[검증 실패] 그래프 분기·필터·관련성·인용 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert paths[True][-2:] == ["generator", "resolve_citations"], '[검증 실패] 그래프 분기·필터·관련성·인용 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert store.search.call_args.kwargs["filters"].must[0].match.value == "법령", '[검증 실패] 그래프 분기·필터·관련성·인용 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("그래프/필터 경계", "설계 한계 확인", "유형 hard filter 전달 확인; 실제 누락 규모 미측정. 스트리밍은 인용 준비 후 외부 생성")

store.search.return_value = []
with patch.object(retrieval, "_get_store", return_value=store), patch.object(gen.llm, "invoke", return_value=AIMessage(content=parsed), side_effect=None) as model:
    empty_state = pipeline.build_graph().invoke({"user_input":"모의 질문"})
assert empty_state["retrieval_passed"] is False and model.call_count == 1, '[검증 실패] 그래프 분기·필터·관련성·인용 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("검색 빈 결과", "정상 경로 확인", '관련성 실패 후 종료, 생성 호출 없음; 이번 모의 검사만으로 검색 0개 원인을 판단하지 않음. 실제 Q02 원인은 8-2절 참조')

passed = relevance.check_relevance({"retrieved_docs":[{"text":"질문과 무관한 모의 문서"}], "similarity_score":1.0}, threshold=0.5)
assert passed["retrieval_passed"] is True, '[검증 실패] 그래프 분기·필터·관련성·인용 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("관련성 판단", "설계 한계 확인", "내용/필수 조건 검증 없이 문서 존재와 점수만으로 통과")

other = {**fixture, "조문번호":"제2조", "version":"다른판본"}
resolved = citations.resolve_citations({"retrieved_docs":[fixture, other]})["citations"]
assert len(resolved) == 1 and resolved[0]["detail"] == "제1조", '[검증 실패] 그래프 분기·필터·관련성·인용 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("인용 중복 제거", "결함 재현", "같은 source_id의 다른 조문·판본이 하나로 합쳐져 둘째 조문 링크 소실")

,노드,입력질문,입력검색어,입력문서수,입력이력수,출력변경키,출력검색통과
0,check_sensitive_info,모의 질문,NaN,0,0,is_sensitive,None
1,understand_query,모의 질문,NaN,0,0,"is_definitive, intent_metadata, needs_link, user_query, user_request",None
2,retrieve,모의 질문,모의 검색어,0,0,"retrieved_docs, similarity_score",None
3,check_relevance,모의 질문,모의 검색어,1,0,"retrieval_passed, fallback_message, is_terminated",True
4,resolve_citations,모의 질문,모의 검색어,1,0,citations,None


,노드,입력질문,입력검색어,입력문서수,입력이력수,출력변경키,출력검색통과
0,check_sensitive_info,모의 질문,NaN,0,0,is_sensitive,None
1,understand_query,모의 질문,NaN,0,0,"is_definitive, intent_metadata, needs_link, user_query, user_request",None
2,retrieve,모의 질문,모의 검색어,0,0,"retrieved_docs, similarity_score",None
3,check_relevance,모의 질문,모의 검색어,1,0,"retrieval_passed, fallback_message, is_terminated",True
4,generator,모의 질문,모의 검색어,1,0,"messages, final_answer",None
5,resolve_citations,모의 질문,모의 검색어,1,1,citations,None


[Mock][설계 한계 확인] 그래프/필터 경계: 유형 hard filter 전달 확인; 실제 누락 규모 미측정. 스트리밍은 인용 준비 후 외부 생성
[Mock][정상 경로 확인] 검색 빈 결과: 관련성 실패 후 종료, 생성 호출 없음; 이번 모의 검사만으로 검색 0개 원인을 판단하지 않음. 실제 Q02 원인은 8-2절 참조
[Mock][설계 한계 확인] 관련성 판단: 내용/필수 조건 검증 없이 문서 존재와 점수만으로 통과
[Mock][결함 재현] 인용 중복 제거: 같은 source_id의 다른 조문·판본이 하나로 합쳐져 둘째 조문 링크 소실


### 3-3. 실패·빈 응답·취소·느린 소비자
모든 생산자는 유한하며 느린 소비자 실험에는 제한 시간을 둔다. 실제 네트워크·LLM 취소를 검증하는 것은 아니다. 종료 이벤트만 있는 빈 답변과, 큐가 가득 찼을 때 토큰/종료 신호 유실을 구분한다.

In [10]:
print("[Mock][시작] 예외·빈 응답·취소·느린 소비자 검사. 아래 서비스 오류 로그는 의도적 실패 입력에 대한 기록입니다.")
with patch.object(chat, "run_preformat", side_effect=RuntimeError('[Mock][의도적 예외] 전처리 실패 시 PIPELINE_FAILED 반환을 검사합니다.')):
    failed = await consume(chat.stream_chat("질문"))
assert failed[-1]["event"] == "error" and failed[-1]["data"]["code"] == "PIPELINE_FAILED", '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("전처리 예외", "정상 경로 확인", "PIPELINE_FAILED 오류 이벤트")

def broken_stream(state):
    """첫 청크 뒤 예외가 발생하는 외부 생성기를 모의한다."""
    yield "일부"
    raise RuntimeError('[Mock][의도적 예외] 생성 중 실패 시 STREAM_FAILED 반환을 검사합니다.')

with patch.object(chat, "run_preformat", return_value={}), patch.object(chat, "stream_formatter", broken_stream):
    failed = await consume(chat.stream_chat("질문"))
assert failed[-1]["data"]["code"] == "STREAM_FAILED" and not any(e["event"] == "message_end" for e in failed), '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("생성 예외", "정상 경로 확인", "STREAM_FAILED 이후 성공 종료 이벤트 없음")

with patch.object(chat, "run_preformat", return_value={}), patch.object(chat, "stream_formatter", return_value=iter(())):
    empty = await consume(chat.stream_chat("질문"))
assert empty[-1]["event"] == "message_end" and not any(e["event"] in {"token","error"} for e in empty), '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("생성 빈 응답", "결함 재현", "답변 없이 성공 종료; Before 수집기는 별도로 EMPTY_RESPONSE 판정")

with patch.object(chat, "run_preformat", return_value={"fallback_message":"추가 정보 필요"}), patch.object(chat, "stream_formatter") as formatter:
    fallback = await consume(chat.stream_chat("질문"))
assert not formatter.called and fallback[-1]["data"]["fallback_triggered"], '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("fallback", "정상 경로 확인", "생성 생략 후 안내와 종료")

# 소비를 잠시 멈춰 용량 64보다 많은 청크를 보내고, 루프 오류를 내용 없이 집계한다.
loop = asyncio.get_running_loop()
loop_errors = []
old_handler = loop.get_exception_handler()
loop.set_exception_handler(lambda loop, ctx: loop_errors.append(type(ctx.get("exception")).__name__))
slow_events = []
timed_out = False
try:
    with patch.object(chat, "run_preformat", return_value={}), patch.object(chat, "stream_formatter", return_value=iter(["x"] * 100)):
        stream = chat.stream_chat("질문")
        slow_events.append(await anext(stream))
        slow_events.append(await anext(stream))
        await asyncio.sleep(0.15)
        try:
            slow_events += await asyncio.wait_for(consume(stream), timeout=0.5)
        except asyncio.TimeoutError:
            timed_out = True
        finally:
            await stream.aclose()
finally:
    loop.set_exception_handler(old_handler)
assert "QueueFull" in loop_errors and timed_out, '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
record("느린 소비자", "결함 재현", "QueueFull 발생·종료 신호 유실로 제한 시간 초과 (100개 모의 청크)")

started, release, finished = threading.Event(), threading.Event(), threading.Event()
def waiting_stream(state):
    """취소 뒤에도 작업 스레드가 남는지 확인하되 종료는 항상 보장한다."""
    started.set()
    try:
        release.wait(timeout=2)
        if False:
            yield ""
    finally:
        finished.set()

with patch.object(chat, "run_preformat", return_value={}), patch.object(chat, "stream_formatter", waiting_stream):
    stream = chat.stream_chat("질문")
    await anext(stream)
    pending = asyncio.create_task(anext(stream))
    try:
        assert await asyncio.to_thread(started.wait, 1), '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
        pending.cancel()
        try:
            await pending
        except asyncio.CancelledError:
            pass
        assert not finished.is_set(), '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
        record("소비자 취소", "결함 재현", "코루틴 취소 후 동기 생성 스레드는 계속 대기; 실제 공급자 취소는 미검증")
    finally:
        release.set()
        assert await asyncio.to_thread(finished.wait, 1), '[검증 실패] 오류 이벤트·빈 응답·취소·느린 소비자 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
        await stream.aclose()
        await asyncio.sleep(0)

[Mock][시작] 예외·빈 응답·취소·느린 소비자 검사. 아래 서비스 오류 로그는 의도적 실패 입력에 대한 기록입니다.
2026-10-20 18:13:41 [error    ] chat_service.preformat_failed  error='[Mock][의도적 예외] 전처리 실패 시 PIPELINE_FAILED 반환을 검사합니다.' exc_type=RuntimeError


[Mock][정상 경로 확인] 전처리 예외: PIPELINE_FAILED 오류 이벤트
[Mock][정상 경로 확인] 생성 예외: STREAM_FAILED 이후 성공 종료 이벤트 없음
[Mock][결함 재현] 생성 빈 응답: 답변 없이 성공 종료; Before 수집기는 별도로 EMPTY_RESPONSE 판정
[Mock][정상 경로 확인] fallback: 생성 생략 후 안내와 종료


[Mock][결함 재현] 느린 소비자: QueueFull 발생·종료 신호 유실로 제한 시간 초과 (100개 모의 청크)
[Mock][결함 재현] 소비자 취소: 코루틴 취소 후 동기 생성 스레드는 계속 대기; 실제 공급자 취소는 미검증


## 4. 판정과 Critical Review
- 정량: 14개 기록의 문서/입력/인용 개수, 본문 포함 개수, 아래 모의 동작 재현 결과를 제시한다. 품질 점수·비용·P95 개선율을 새로 만들지 않는다.
- 정성: 질문별 기존 검토 의견과 기계적 관찰을 함께 읽는다. 입력에 근거가 있다는 이유만으로 답변 오류를 전부 생성 모델 탓으로 돌리지 않는다.
- 채택: 기존 기록 분석 + 외부 경계 모의 방식. 실제 함수를 사용하므로 별도 서비스 복사본을 만들지 않는다.
- 수정 제안: 이력/계약 전달과 답변 이력, 조문·판본별 인용, 빈 응답, SSE 생산/취소 계약을 각각 독립 변경으로 검토한다.
- 미확정: 실제 필터가 Q02를 비웠는지, 요약이 검색 품질을 얼마나 낮췄는지, 날짜 메타데이터 추가가 답변을 개선하는지. 유료 비교 전에 질문·한 변수·성공 기준을 정해야 한다.
- 라이브러리 자체 결함을 입증한 결과는 없다. 여기서는 서비스 연결 코드의 재현과 설계 한계를 관찰했다.
- 개선/회귀 사례는 수정 전이므로 각각 0개. 미해결 대표 사례는 Q02(검색 0개 원인), Q05(요약 손실의 영향), T02(판본 적용 오류의 원인). 기존 심각한 오류 의견 전체는 14개 분석 및 03 검토 노트북을 유지한다.
- 누수: 평가 라벨·검토 정답을 서비스 입력에 넣지 않았다. 모의 fixture는 성능 평가셋과 분리했다. 독립 holdout 성능을 주장하지 않는다.
- 공개 범위: 전체 원문·프롬프트는 출력하지 않는다. 개인 데이터 없는 모의값만 사용한다. 원시 응답·점수·서비스 파일은 보존한다.

다음 작업은 **결함별 최소 수정안과 회귀 기준 제시 → 사용자 승인**이다. 이번 승인만으로 서비스 수정·GPU 실행·commit/PR·merge를 진행하지 않는다.

실행 준비 중 첫 검증은 app.__init__의 무관한 API import가 모의 벡터 모듈의 하위 모듈을 요구해 실패했다. 테스트에서 app 패키지 경로만 제공하도록 좁혀 해결했으며 서비스 구현은 수정하지 않았다. 이 import 실패는 서비스 품질 결함으로 집계하지 않는다.
두 번째 검증에서는 patch.dict 종료 후 제거된 모듈 때문에 LangGraph의 타입 참조 해석이 실패했다. 전용 커널 동안 모의 모듈을 유지하도록 수정했다. 다른 작업은 커널을 재시작한 뒤 수행한다.

### 기존 기록 분석의 결론
생성 입력이 기록된 13개 질문 모두 검색 문서 본문 5개가 문자열 기준으로 포함됐다(65/65). 따라서 이 실행에서 **검색 본문 전체가 전달되지 않았다**는 주장은 지지되지 않는다. 다만 조문·판본 메타데이터의 별도 전달이나 모델의 올바른 해석은 별개다.
Q04의 민법 제147조, Q09의 제565조·제567조, Q11의 형법 제319조는 코퍼스에 별도 문서가 있다. 이 질문의 검색 5개 목록에는 없으므로, 이 사례들을 전부 코퍼스 부재로 설명할 수 없다. 판본 적합성은 아직 미확정이다. Q09는 같은 조문의 다른 판본 5개가 검색되고 인용은 1개여서 다양성과 판본 구분을 별도로 검토해야 한다.

### 수정 승인 시 사용할 검증 기준
| 수정 후보 | 통과해야 할 기준 | 주의점 |
|---|---|---|
| 이력·계약 정보 전달 | 모의 입력 표식이 의도한 모델 입력에 도달 | 길이·민감정보 처리와 이력 중복을 함께 설계 |
| 생성 답변 이력 | 사용자/AI 메시지 역할과 순서 보존 | 호출자의 세션 저장 책임도 확인 |
| 인용 식별 | 같은 법령의 다른 조문·판본 구분 | 같은 근거의 중복 청크는 중복 표시하지 않음 |
| 빈 응답 | 빈 생성에 성공 종료를 보내지 않음 | fallback 정상 응답과 구분 |
| SSE 흐름 제어 | 느린 소비에도 청크/종료 유실 없음 | 큐 크기 확대만으로 해결했다고 하지 않음 |
| 취소 | 소비자 종료 후 생산 종료 및 자원 정리 확인 | 실제 모델 공급자의 취소 지원은 별도 통합 확인 |

이 표는 수정 제안이며 이번에 구현하지 않았다. 모의 검사 13개 중 결함 재현 6개, 설계 한계 3개, 정상 경로 4개다. 판본 적합성·법률 내용은 새로운 판단을 내리지 않았다.

In [11]:
display(pd.DataFrame(checks))
assert len(checks) == 13, '[검증 실패] 모의 검사 수·원본 보존·실제 호출 격리: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == value for p, value in before_hashes.items()), '[검증 실패] 모의 검사 수·원본 보존·실제 호출 격리: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert model_module.streaming_llm.stream.call_count == 0, '[검증 실패] 모의 검사 수·원본 보존·실제 호출 격리: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
print('[Mock][검증 완료] 13개 검사에서 예상 동작을 확인했습니다. 결함 재현 통과는 결함 수정 완료를 뜻하지 않습니다.\n[검증][완료] Before·코퍼스·사용자 검토 해시 불변. 실제 모델·검색 호출 없음.')

,검사,분류,관찰
0,이력/계약 정보 전달,결함 재현,서비스가 받은 history와 contract_context가 다음 경계에 없음
1,생성 답변의 이력 저장,결함 재현,final_answer는 반환하지만 messages 갱신에는 사용자 질문만 있음
2,문맥 메타데이터,설계 한계 확인,"본문·제목·유형 전달, 별도 version/effective_from 필드는 미전달; 본문 내 날짜까지 없다는 뜻은 아님"
3,그래프/필터 경계,설계 한계 확인,유형 hard filter 전달 확인; 실제 누락 규모 미측정. 스트리밍은 인용 준비 후 외부 생성
4,검색 빈 결과,정상 경로 확인,"관련성 실패 후 종료, 생성 호출 없음; 이번 모의 검사만으로 검색 0개 원인을 판단하지 않음. 실제 Q02 원인은 8-2절 참조"
5,관련성 판단,설계 한계 확인,내용/필수 조건 검증 없이 문서 존재와 점수만으로 통과
6,인용 중복 제거,결함 재현,같은 source_id의 다른 조문·판본이 하나로 합쳐져 둘째 조문 링크 소실
7,전처리 예외,정상 경로 확인,PIPELINE_FAILED 오류 이벤트
8,생성 예외,정상 경로 확인,STREAM_FAILED 이후 성공 종료 이벤트 없음
9,생성 빈 응답,결함 재현,답변 없이 성공 종료; Before 수집기는 별도로 EMPTY_RESPONSE 판정


[Mock][검증 완료] 13개 검사에서 예상 동작을 확인했습니다. 결함 재현 통과는 결함 수정 완료를 뜻하지 않습니다.
[검증][완료] Before·코퍼스·사용자 검토 해시 불변. 실제 모델·검색 호출 없음.


## 5. 모든 활성 노드 점검표 — 부분 완료를 전체 완료로 표시하지 않기
모든 활성 노드를 코드로 읽고, 로컬 검증과 실제 기록 검증을 구분한다. 원문 질문 14개가 각 노드의 모든 분기를 덮는 것은 아니므로 차단·파싱 오류 등은 별도 모의 입력으로 확인한다. 실제 실행에서도 지나가지 않은 노드는 `not_reached`로 남긴다.

`formatter`는 노드로 호출되지 않지만 인용 섹션 생성 helper는 활성이다. 과거 `intent_understanding`, `query_summary`, `expression_revision`은 현재 그래프 등록 대상이 아니다. 파일은 삭제하지 않는다.

아래 표는 실제 수집 전의 점검표다. 2026-10-20 수집 후 관측 횟수와 남은 한계는 8절 표로 갱신한다.

In [12]:
active = {name for name in pipeline.build_graph().nodes if not name.startswith("__")}
assert active == {"check_sensitive_info", "understand_query", "retrieve", "check_relevance", "generator", "resolve_citations"}, '[검증 실패] 활성 노드 구성: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
audit_rows = [
 ("check_sensitive_info", "정규식 차단과 분기", "14개 질문·합성 민감패턴 검사", "원 실행의 출력 미기록; 실제 노드 기록 필요"),
 ("understand_query", "질문 요약·유형·확답 분기", "JSON 오류·부정문·타입 검사", "요약만 남음; 원 모델 출력/필터/판정 필요"),
 ("retrieve", "질의·필터·검색 문서", "필터 전달·빈 결과", "65개 문서 존재; 필터 미기록·관련도 검토 필요"),
 ("check_relevance", "통과/유보", "임계값 경계·문서 없음", "통과 여부 기록 있음; 당시 입력 점수 연결 필요"),
 ("generator", "실제 메시지와 생성 답변", "문맥·이력·메타데이터·출력 역할", "Before 답변/빌더 기록 있음; 실제 모델 호출 메시지 관찰 필요"),
 ("resolve_citations", "근거 단위·판본·URL", "다른 조문 합침·빈 입력", "기존 인용 있음; 모델 주장과 출처 지지 관계 미확인"),
 ("service/stream_formatter", "서비스 입력·토큰·인용·종료", "실패·빈 응답·취소·느린 소비자", "기존 SSE 있음; 원격 계측 경로 동일성 확인 필요"),
]
display(pd.DataFrame(audit_rows, columns=["노드/경계","역할","로컬검증","실제검증/남은일"]))

,노드/경계,역할,로컬검증,실제검증/남은일
0,check_sensitive_info,정규식 차단과 분기,14개 질문·합성 민감패턴 검사,원 실행의 출력 미기록; 실제 노드 기록 필요
1,understand_query,질문 요약·유형·확답 분기,JSON 오류·부정문·타입 검사,요약만 남음; 원 모델 출력/필터/판정 필요
2,retrieve,질의·필터·검색 문서,필터 전달·빈 결과,65개 문서 존재; 필터 미기록·관련도 검토 필요
3,check_relevance,통과/유보,임계값 경계·문서 없음,통과 여부 기록 있음; 당시 입력 점수 연결 필요
4,generator,실제 메시지와 생성 답변,문맥·이력·메타데이터·출력 역할,Before 답변/빌더 기록 있음; 실제 모델 호출 메시지 관찰 필요
5,resolve_citations,근거 단위·판본·URL,다른 조문 합침·빈 입력,기존 인용 있음; 모델 주장과 출처 지지 관계 미확인
6,service/stream_formatter,서비스 입력·토큰·인용·종료,실패·빈 응답·취소·느린 소비자,기존 SSE 있음; 원격 계측 경로 동일성 확인 필요


### 5-1. 빠져 있던 민감정보·질문 이해·관련성 분기 검사
합성 번호는 테스트 안에서만 만들고 출력하지 않는다. 문장이 안전한지 법률적으로 판단하는 검사가 아니라 정규식/JSON/분기 계약 검사다. 모델의 실제 민감정보·의도 분류 품질은 별도 기록이 필요하다.

In [13]:
print("[Mock][시작] 합성 민감정보·잘못된 JSON·타입·부정문 검사. 아래 차단 안내와 warning/info 로그는 테스트 입력에서 발생합니다.")
import app.graph.nodes.check_sensitive_info as sensitive
extra_checks = []
normal = [sensitive.check_sensitive_info({"user_input": q["question"]})["is_sensitive"] for q in items]
assert not any(normal), '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
synthetic = " ".join(["0" * 4] * 4)
blocked = sensitive.check_sensitive_info({"user_input": synthetic})
assert blocked["is_sensitive"] and blocked["fallback_message"], '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
# 차단 분기에서는 의도 모델/검색을 호출하면 안 된다.
with patch.object(gen.llm, "invoke", side_effect=AssertionError('[Mock][검증 실패] 민감정보 차단 후 모델이 호출됐습니다. 차단 노드의 종료 분기를 확인하세요.')):
    stopped = pipeline.build_graph().invoke({"user_input": synthetic})
assert stopped["is_sensitive"] and "retrieved_docs" not in stopped, '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
extra_checks.append(("민감정보", "14개 원 질문 통과·합성 카드 패턴 차단·모델 호출 생략"))

with patch.object(gen.llm, "invoke", return_value=AIMessage(content="not-json"), side_effect=None):
    invalid = understand.understand_query({"user_input":"모의 원 질문"})
assert invalid["user_query"] == "모의 원 질문" and invalid["intent_metadata"]["doc_type"] is None, '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
extra_checks.append(("JSON 파싱 실패", "원 질문·필터 없음으로 fallback; 오류 로그는 원문 일부를 포함할 수 있음"))

for raw, expected in [("[]", "AttributeError"), ('{"is_definitive":"false"}', "blocked")]:
    with patch.object(gen.llm, "invoke", return_value=AIMessage(content=raw), side_effect=None):
        try:
            result = understand.understand_query({"user_input":"모의 원 질문"})
            outcome = "blocked" if result["is_definitive"] else "passed"
        except AttributeError:
            outcome = "AttributeError"
    assert outcome == expected, '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
    extra_checks.append(("JSON 스키마 " + raw, outcome + "; 실제 모델이 이 형식을 냈다는 증거는 아직 없음"))

with patch.object(gen.llm, "invoke", return_value=AIMessage(content='{"is_definitive":false}'), side_effect=None):
    negated = understand.understand_query({"user_input":"확답을 원하지 않습니다. 확인할 조건을 알려주세요."})
assert negated["is_definitive"], '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
extra_checks.append(("확답 부정문", "LLM false여도 정규식 매칭으로 차단; 정상 요청 오탐 재현"))

for docs, score, expected in [([],1.0,False),([doc],0.49,False),([doc],0.5,True)]:
    result = relevance.check_relevance({"retrieved_docs":docs,"similarity_score":score}, threshold=0.5)
    assert result["retrieval_passed"] == expected, '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
extra_checks.append(("관련성 경계", "빈 문서/임계값 미만 거절, 임계값 동일 통과"))
assert citations.resolve_citations({"retrieved_docs":[]}) == {"citations":[]}, '[검증 실패] 민감정보·JSON·관련성 경계 재현: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
display(pd.DataFrame(extra_checks, columns=["검사", "관찰"]))
print("[Mock][검증 완료] 추가 분기 검사에서 예상 결과를 확인했습니다. 실제 모델의 오류 발생률을 측정한 것은 아닙니다.")

[Mock][시작] 합성 민감정보·잘못된 JSON·타입·부정문 검사. 아래 차단 안내와 warning/info 로그는 테스트 입력에서 발생합니다.
<<주의>>
카드번호로 보이는 정보가 포함되어 있습니다.
민감한 정보가 포함되어있지 않나요? 다시 한 번 확인해주세요!

예) 주민등록번호, 계좌번호와 같은 개인정보/민감정보는 보안상 입력하지 않도록 권장합니다.
2026-10-20 18:13:41 [warning  ] understand_query.json_parse_failed raw=t- user_input='모의 원 질문'


2026-10-20 18:13:41 [info     ] understand_query.definitive_blocked llm_definitive=True matched_pattern=None user_input='모의 원 질문'


2026-10-20 18:13:41 [info     ] understand_query.definitive_blocked llm_definitive=False matched_pattern=확답 user_input='확답을 원하지 않습니다. 확인할 조건을 알려주세요.'


,검사,관찰
0,민감정보,14개 원 질문 통과·합성 카드 패턴 차단·모델 호출 생략
1,JSON 파싱 실패,원 질문·필터 없음으로 fallback; 오류 로그는 원문 일부를 포함할 수 있음
2,JSON 스키마 [],AttributeError; 실제 모델이 이 형식을 냈다는 증거는 아직 없음
3,"JSON 스키마 {""is_definitive"":""false""}",blocked; 실제 모델이 이 형식을 냈다는 증거는 아직 없음
4,확답 부정문,LLM false여도 정규식 매칭으로 차단; 정상 요청 오탐 재현
5,관련성 경계,"빈 문서/임계값 미만 거절, 임계값 동일 통과"


[Mock][검증 완료] 추가 분기 검사에서 예상 결과를 확인했습니다. 실제 모델의 오류 발생률을 측정한 것은 아닙니다.


## 6. 실제 검색 결과를 질문·근거와 대조
모의 검색 결과는 검색 품질을 대신하지 않는다. 아래는 실제 Before의 65개 후보를 판본·조문별로 보여준다. `gold 조문 후보`는 이름/조문이 같은 후보일 뿐 정답 인정이나 관련도 점수가 아니다. 적용 시점·부칙·문서 내용을 검토하지 않고 Recall/nDCG를 계산하지 않는다. 검토가 필요한 목록을 명확히 드러내는 단계다.

검색 품질 검토 순서: 코퍼스 보유 → 실제 회수 → 판본 적합성 → 문서 관련도(0/1/2) → 필수 근거 누락 → 전달된 근거와 답변의 주장 대응. 기존 답변의 5점 점수를 검색 관련도 점수로 변환하지 않는다.

In [14]:
retrieval_review = []
for q in items:
    qid = q["id"]
    rule = q["evaluation"]["retrieval"]
    for rank, d in enumerate(answers[qid]["trace"].get("retrieved_docs") or [], 1):
        candidates = [g["source_id"] for g in q["gold_sources"]
                      if g.get("type") == "law" and g.get("law_name") == d.get("title")
                      and g.get("article") == d.get("조문번호")]
        retrieval_review.append({"ID":qid, "순위":rank, "문서":d.get("title"),
            "조문":d.get("조문번호"), "판본":d.get("version"), "유사도":d.get("score"),
            "gold 조문 후보":", ".join(candidates), "관련도(사람검토)":None,
            "점수보류":rule.get("scoring_status", "판본/관련도 검토 필요")})
display(pd.DataFrame(retrieval_review))
print('[기록][안내] Q02는 검색 문서가 0개이므로 후보 표에 행이 없습니다. 질문 기록이 누락된 것은 아닙니다.')

def inspect_saved_case(qid):
    """질문 ID를 받아 비공개 원문을 대조용으로 표시한다. 원문 출력은 Git에 저장하지 않는다."""
    record = answers[qid]
    print("질문:", record["question"])
    print("검색어:", record["trace"].get("user_query"))
    for rank, d in enumerate(record["trace"].get("retrieved_docs") or [], 1):
        print("\n근거", rank, d.get("version"), d.get("조문번호"), "\n", d.get("text"))
    print("\n답변:\n", record["answer"])
    print("\n기존 검토 의견:\n", notes[qid])

# 예: inspect_saved_case("Q11") — 실행 후 원문 출력은 공개 저장 전에 지운다.

,ID,순위,문서,조문,판본,유사도,gold 조문 후보,관련도(사람검토),점수보류
0,Q01,1,"대법원 2020다279456, 279463 건물명도(인도)·임대차보증금",NaN,"판례|2020다279456, 279463",0.622051,,None,pending_case_selection
1,Q01,2,민법,제635조,민법|제635조@2026-03-17,0.615482,Q01-S02,None,pending_case_selection
2,Q01,3,주택임대차보호법,제6조의2,주택임대차보호법|제6조의2@2008-03-21,0.611864,Q01-S03,None,pending_case_selection
3,Q01,4,대법원 2017다260636 임대차보증금,NaN,판례|2017다260636,0.610873,,None,pending_case_selection
4,Q01,5,"대법원 2023다236566, 236573 임차보증금반환·차임연체료및손해배상금",NaN,"판례|2023다236566, 236573",0.609853,,None,pending_case_selection
...,...,...,...,...,...,...,...,...,...
60,T02,1,법령해석례 22-0680 민원인 - 100세대 이상의 공동주택을 임대하는 임대사업자가 종전 임차인과의 임대차계약이 만료되어 새로운 임차인과 임대차계약을 체결하는 경우 임대차계약의 신고 기한(「민간임대주택에 관한 특별법」 제46조 등 관련),NaN,해석례|22-0680,0.596076,,None,판본/관련도 검토 필요
61,T02,2,법령해석례 20-0049 민원인 - 임대사업자 등록 전에 체결한 임대차계약의 임대료를 임대사업자 등록 후 신고한 경우 신고한 임대료가 최초 임대료에 해당하는지 여부(법률 제16000호 「민간임대주택에 관한 특별법」 제44조 등 관련),NaN,해석례|20-0049,0.595295,,None,판본/관련도 검토 필요
62,T02,3,부동산 거래신고 등에 관한 법률,제6조의2,부동산 거래신고 등에 관한 법률|제6조의2@2021-06-01,0.594801,T02-S01,None,판본/관련도 검토 필요
63,T02,4,주택임대차보호법,제3조의7,주택임대차보호법|제3조의7@2023-04-18,0.587132,,None,판본/관련도 검토 필요


[기록][안내] Q02는 검색 문서가 0개이므로 후보 표에 행이 없습니다. 질문 기록이 누락된 것은 아닙니다.


In [15]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert len(retrieval_review) == 65, '[검증 실패] Before 검색 후보 개수: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

## 2026-10-20 실제 실행 준비 기록
사용자가 2시간·GPU $1.18 이내, 백업 후 중지를 승인했다. 종료 경계는 17:54:57 KST이며 준비 시간도 포함한다. A40·기존 vLLM digest·EXAONE revision·BF16·32768을 확인했다. 이전 volume의 코퍼스·색인은 남아 있지만 가상환경 일부 파일이 없어 별도 CPU venv를 고정 버전으로 복구했다. 서비스 변경·재색인·모델 변경이 아니다. 실제 결과는 수집 후 아래에 추가하며, 준비 완료를 실행 성공으로 취급하지 않는다.

### 실제 실행 준비 실패와 복구
2026-10-20 첫 시도는 코퍼스·서비스·패키지·색인 2494개 검증까지 통과했으나, 질문 루프 전에 runtime.json을 저장하는 과정에서 NameError(json)가 발생했다. 기존 03에서 추출한 harness.before가 json import를 누락했다. 03 원본 export 셀에 import json만 추가하고 추출했다. 서비스·프롬프트·모델은 바꾸지 않았다. 실패한 실행 노트북과 빈 runtime 파일은 보존하며 다른 결과 디렉터리에서 다시 시작한다. 첫 시도에는 실제 질문 호출이 없었다.

In [16]:
from tempfile import TemporaryDirectory
from harness.before import save_trace
with TemporaryDirectory(prefix="aigo-audit-save-") as temporary:
    target = Path(temporary) / "runtime.json"
    save_trace(target, {"fixture": True})
    assert json.loads(target.read_text(encoding="utf-8")) == {"fixture": True}, '[검증 실패] JSON 저장 결과: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
    try:
        save_trace(target, {"overwrite": True})
    except FileExistsError:
        pass
    else:
        raise AssertionError('[검증][실패] 기존 결과 저장 시 FileExistsError가 발생하지 않았습니다. save_trace의 덮어쓰기 방지 동작을 확인하세요.')
print('[검증][완료] 독립 모듈의 JSON 저장과 기존 파일 덮어쓰기 거부를 확인했습니다.')

[검증][완료] 독립 모듈의 JSON 저장과 기존 파일 덮어쓰기 거부를 확인했습니다.


## 8. 실제 14개 실행 결과 — 2026-10-20
**문제:** 이전 Before에는 일부 중간 상태가 없어 원인을 구분할 수 없었다. **방법:** 기존 코드·코퍼스·모델을 유지하고 질문마다 실제 노드 입력/출력을 기록했다. 이는 개선 후 성능 실험이 아닌 `node_audit_not_before`다.

원격 실행 완료와 14개 파일을 확인한 뒤 백업 해시를 대조하고 Pod의 `EXITED`를 확인했다. 아래 셀은 GPU 없이 백업을 읽는다. 원문은 Git 밖 캐시에 복구하고, 공개 출력에는 요약만 남긴다. 백업이 없는 다른 PC에서는 백업 파일을 먼저 복사해야 한다. 해시만으로 원문을 복구할 수는 없다.

실패한 첫 시도와 원격 실행 노트북도 압축파일에 보존했다. 아래 복구는 알려진 결과 파일만 읽고 기존 파일은 덮어쓰지 않는다.

In [17]:
import tarfile
AUDIT_SHA = "4ded076ebafb1012987524e34ba329fb8a1e8902d259d43300369fdd36414171"
archive = ROOT / "db/runpod-transfer/node-audit-20261020.tar.gz"
if not (archive.is_file()):
    raise FileNotFoundError('[기록][파일 없음] db/runpod-transfer/node-audit-20261020.tar.gz 백업이 필요합니다. 검증된 비공개 백업을 복사한 뒤 다시 실행하세요.')
if not (hashlib.sha256(archive.read_bytes()).hexdigest() == AUDIT_SHA):
    raise ValueError('[검증 실패] 백업 해시·실행 종류·원문 질문·코퍼스·평가셋 일치: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.')
AUDIT_DIR = CORPUS / "node_audits" / "20261020"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive) as backup:
    for name in ["runtime.json", *[f"{q['id']}.json" for q in items]]:
        data = backup.extractfile(f"audit-20261020/results-v2/{name}").read()
        target = AUDIT_DIR / name
        if target.exists():
            if not (target.read_bytes() == data):
                raise ValueError(f'기존 기록과 다름: {name}')
        else:
            target.write_bytes(data)
audit = {q["id"]: json.loads((AUDIT_DIR / f"{q['id']}.json").read_text(encoding="utf-8")) for q in items}
audit_runtime = json.loads((AUDIT_DIR / "runtime.json").read_text(encoding="utf-8"))
assert audit_runtime["corpus_sha256"] == manifest["corpus_sha256"], '[검증 실패] 백업 해시·실행 종류·원문 질문·코퍼스·평가셋 일치: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert audit_runtime["eval_sha256"] == hashlib.sha256((ROOT / "notebooks/redesign/eval/eval_set.json").read_bytes()).hexdigest(), '[검증 실패] 백업 해시·실행 종류·원문 질문·코퍼스·평가셋 일치: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(r["run_kind"] == "node_audit_not_before" and r["status"] == "ok" for r in audit.values()), '[검증 실패] 백업 해시·실행 종류·원문 질문·코퍼스·평가셋 일치: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(audit[q["id"]]["question"] == q["question"] for q in items), '[검증 실패] 백업 해시·실행 종류·원문 질문·코퍼스·평가셋 일치: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
node_records = {qid: {x["node"]: x for x in r["trace"]["nodes"]} for qid, r in audit.items()}
print('[기록][완료] 백업 해시·14개 질문 일치 확인 및 비공개 캐시 복구 완료. status=ok는 요청 완료이며 정확성 점수가 아닙니다.')

[기록][완료] 백업 해시·14개 질문 일치 확인 및 비공개 캐시 복구 완료. status=ok는 요청 완료이며 정확성 점수가 아닙니다.


### 8-1. 질문별 흐름과 노드별 관측 횟수
`통과`는 현재 관련성 노드가 답변 생성을 허용했다는 의미다. 정답 여부가 아니다. 시간은 원격 계측을 포함한 관측값이며 Before 대비 개선율로 사용하지 않는다. 노드별 미도달은 오류와 구분한다. 실제 서비스는 `generator_stream`을 호출하며 비스트리밍 `generator`는 앞의 모의 검증 범위다.

In [18]:
audit_rows = []
for qid, record in audit.items():
    nodes = node_records[qid]
    intent = nodes["understand_query"]["output"]
    found = nodes["retrieve"]["output"]
    audit_rows.append({"ID": qid, "검색 분류": intent["intent_metadata"].get("doc_type"),
        "검색어": intent["user_query"], "문서 수": len(found["retrieved_docs"]),
        "최고 유사도": round(found["similarity_score"], 3),
        "통과": nodes["check_relevance"]["output"]["retrieval_passed"],
        "인용 수": len(record["citations"]), "전체 초": round(record["total_ms"] / 1000, 2),
        "첫 토큰 초": round(record["ttft_ms"] / 1000, 2)})
display(pd.DataFrame(audit_rows))
coverage = []
for name in audit["Q01"]["node_status"]:
    observed = [x[name] for x in node_records.values() if name in x]
    coverage.append({"노드": name, "관측/14": len(observed), "미도달/14": 14-len(observed),
        "노드 오류 수": sum(x["error"] is not None for x in observed),
        "중앙 시간 ms": round(pd.Series([x["elapsed_ms"] for x in observed]).median(), 2)})
display(pd.DataFrame(coverage))

,ID,검색 분류,검색어,문서 수,최고 유사도,통과,인용 수,전체 초,첫 토큰 초
0,Q01,NaN,월세 계약 중간 해지 절차,5,0.622,True,5,20.09,5.71
1,Q02,법률해석례,전세 계약 만료 전 특약 전세권 설정으로 인한 집주인 연락 두절 시 계약 파기 가능 여부,0,0.000,False,0,3.27,3.27
2,Q03,판례,계약 해지 사유로 귀신 출현 가능한가,5,0.516,True,5,22.09,8.91
3,Q04,NaN,전세자금 대출 승인 후에 임대차 계약 효력 발생 특약 가능 여부,5,0.631,True,5,17.52,4.74
4,Q05,NaN,특약 작성 방법,5,0.503,True,2,18.47,2.47
5,Q06,법령,임대차 계약서에서 '차임'의 의미,5,0.665,True,2,6.32,2.60
6,Q07,NaN,윗집 소음 해결 방법,5,0.530,True,2,14.74,2.58
7,Q08,NaN,공인중개사 준비 방법,5,0.493,True,3,13.76,3.20
8,Q09,법령,계약 무산 시 선지불 보증금 반환 가능 여부,5,0.619,True,1,14.10,3.67
9,Q10,NaN,자취생 주거비 지원 정책,5,0.463,True,4,13.47,3.50


,노드,관측/14,미도달/14,노드 오류 수,중앙 시간 ms
0,check_sensitive_info,14,0,0,0.09
1,understand_query,14,0,0,2464.35
2,retrieve,14,0,0,191.51
3,check_relevance,14,0,0,0.01
4,resolve_citations,13,1,0,0.07
5,generator_stream,13,1,0,11162.21


In [19]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert sum(x["문서 수"] for x in audit_rows) == 65, '[검증 실패] 실제 노드 관측 수·오류 상태: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert len(node_records["Q02"]) == 4, '[검증 실패] 실제 노드 관측 수·오류 상태: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(x["error"] is None for nodes in node_records.values() for x in nodes.values()), '[검증 실패] 실제 노드 관측 수·오류 상태: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

### 8-2. Q02: 잘못된 분류가 검색 대상을 모두 제외함
**쉬운 설명:** 서가 이름이 ‘법령해석례’인데 ‘법률해석례’ 서가에서만 찾으라고 지시했다. 문서가 있어도 이 조건으로는 후보가 될 수 없다.

실제 모델 원응답 → 질문 이해 노드 출력 → 검색 노드 입력에서 같은 값이 확인된다. `_build_filter`는 이를 그대로 `MatchValue`에 넣는다. [Qdrant 공식 필터 문서](https://qdrant.tech/documentation/search/filtering/)의 문자열 일치 조건과 실제 서비스 코드를 근거로, 고정 코퍼스의 서비스용 메타데이터를 대조한다.

비교는 **잘못된 이름 / 실제 분류명 / 필터 없음**의 후보 자격 수다. 임베딩을 다시 계산하거나 정답 검색을 수행한 실험이 아니다. 바른 이름이 Q02에 적합한 분류라는 뜻도 아니다. 이 결과는 이번 수집의 0건 원인을 설명하지만 당시 필터가 없는 과거 Before의 원인까지 확정하지 않는다.

In [20]:
from collections import Counter
from harness.before import service_payload
corpus_docs = [json.loads(line) for line in (CORPUS / "corpus.jsonl").read_text(encoding="utf-8").splitlines()]
types = Counter(service_payload(d)["doc_type"] for d in corpus_docs)
q02 = node_records["Q02"]
metadata = q02["retrieve"]["input"]["intent_metadata"]
condition = retrieval._build_filter(metadata).must[0]
display(pd.DataFrame([
    {"조건": condition.match.value, "후보 자격 문서 수": types[condition.match.value]},
    {"조건": "법령해석례", "후보 자격 문서 수": types["법령해석례"]},
    {"조건": "분류 필터 없음", "후보 자격 문서 수": sum(types.values())}]))
print('[기록][원인 확인] Q02: 모델이 생성한 분류명의 검증 누락으로 후보 자격 문서가 0개입니다. 수정 후 검색 품질은 미측정입니다.')

,조건,후보 자격 문서 수
0,법률해석례,0
1,법령해석례,55
2,분류 필터 없음,2494


[기록][원인 확인] Q02: 모델이 생성한 분류명의 검증 누락으로 후보 자격 문서가 0개입니다. 수정 후 검색 품질은 미측정입니다.


In [21]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert metadata == q02["understand_query"]["output"]["intent_metadata"], '[검증 실패] Q02 분류 전달·필터·후보 자격 수: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert condition.key == "doc_type" and condition.match.value == "법률해석례", '[검증 실패] Q02 분류 전달·필터·후보 자격 수: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert types[condition.match.value] == 0 and types["법령해석례"] == 55, '[검증 실패] Q02 분류 전달·필터·후보 자격 수: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert not q02["retrieve"]["output"]["retrieved_docs"], '[검증 실패] Q02 분류 전달·필터·후보 자격 수: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

### 8-3. 검색 조건 손실·판본 집중·생성 문맥 전달
Q05 검색어는 이번에도 ‘특약 작성 방법’이다. 원문의 고양이·임차인 부담 조건을 검색어에 반영하지 않았다. 원문을 생성 단계에서 볼 수 있더라도 이미 빠진 검색 근거가 자동으로 복구되지는 않는다.

Q09는 같은 조문의 다른 판본 5개가 검색되었고 인용은 1개다. 현재 인용 중복 제거는 `source_id`만 사용한다. 아래는 실제 문서의 판본과 출력 개수를 대조한다. 판본을 합쳤다는 사실과 어떤 판본이 정답인지는 다른 판단이다.

문맥 검사는 검색 본문이 실제 생성 메시지에 포함되는지만 본다. 주장의 정확성·인용 지지 여부는 사람 검토가 필요하다.

In [22]:
q05_query = node_records["Q05"]["understand_query"]["output"]["user_query"]
q09_docs = node_records["Q09"]["retrieve"]["output"]["retrieved_docs"]
display(pd.DataFrame([{k: d.get(k) for k in ["title", "조문번호", "version", "source_id"]} for d in q09_docs]))
context_rows = []
for qid, r in audit.items():
    docs = node_records[qid]["retrieve"]["output"]["retrieved_docs"]
    messages = r["trace"]["generation_messages"] or []
    context = "\n".join(str(m["content"]) for m in messages)
    context_rows.append({"ID": qid, "검색 본문 수": len(docs),
        "생성 메시지 내 본문 수": sum(d["text"] in context for d in docs)})
display(pd.DataFrame(context_rows))
# 의도 분석 호출의 실제 usage만 집계한다. 생성 호출을 포함한 전체 usage로 부르지 않는다.
intent_usage = [r["trace"]["intent_response"]["usage_metadata"] for r in audit.values()]
print('[기록][사용량] 의도 분석 호출에서 반환된 total_tokens 합:', sum(u["total_tokens"] for u in intent_usage))
print('[기록][미측정] 생성 포함 전체 usage·GPU 실청구 비용은 unknown입니다. 문자 수를 토큰 수로 대체하지 않습니다.')

,title,조문번호,version,source_id
0,주택임대차보호법,제3조의2,주택임대차보호법|제3조의2@2013-08-13,001248
1,주택임대차보호법,제3조의2,주택임대차보호법|제3조의2@2015-07-01,001248
2,주택임대차보호법,제3조의2,주택임대차보호법|제3조의2@2002-06-30,001248
3,주택임대차보호법,제3조의2,주택임대차보호법|제3조의2@2016-12-01,001248
4,주택임대차보호법,제3조의2,주택임대차보호법|제3조의2@2002-07-01,001248


,ID,검색 본문 수,생성 메시지 내 본문 수
0,Q01,5,5
1,Q02,0,0
2,Q03,5,5
3,Q04,5,5
4,Q05,5,5
5,Q06,5,5
6,Q07,5,5
7,Q08,5,5
8,Q09,5,5
9,Q10,5,5


[기록][사용량] 의도 분석 호출에서 반환된 total_tokens 합: 8690
[기록][미측정] 생성 포함 전체 usage·GPU 실청구 비용은 unknown입니다. 문자 수를 토큰 수로 대체하지 않습니다.


In [23]:
# 검증: 위 분석 결과와 고정 기록의 일치 확인
assert q05_query == "특약 작성 방법", '[검증 실패] Q05 검색어·Q09 판본·생성 문맥 전달: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert len(q09_docs) == 5 and len(audit["Q09"]["citations"]) == 1, '[검증 실패] Q05 검색어·Q09 판본·생성 문맥 전달: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert len({d.get("version") for d in q09_docs}) == 5, '[검증 실패] Q05 검색어·Q09 판본·생성 문맥 전달: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert sum(x["생성 메시지 내 본문 수"] for x in context_rows) == 65, '[검증 실패] Q05 검색어·Q09 판본·생성 문맥 전달: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'

### 8-4. 노드별 판정과 다음 수정 순서
| 노드/경계 | 실제 확인 | 남은 문제·판정 |
|---|---|---|
| 민감정보 | 14개 모두 통과 | 합성 차단은 모의 검증. 민감정보 탐지 전체 정확도는 미측정 |
| 질문 이해 | 14개 원응답·출력 확보 | Q02 잘못된 분류, Q05 조건 손실. JSON 타입/부정문 결함은 모의 재현과 구분 |
| 검색 | Q02 0개, 나머지 각 5개 | 필터 검증 우선. Q09 판본 집중과 관련도는 후속 검색 비교 대상 |
| 관련성 | 13개 통과, Q02 유보 | top-1 점수 통과가 필수 근거 확보를 뜻하지 않음. Q08·Q10도 통과함 |
| 생성 | 스트리밍 13개, Q02 미도달 | 본문 65/65 전달. 날짜 메타데이터·다회차 이력 문제와 법률적 답변 품질은 별개 |
| 인용 | 13개 관측 | Q09 판본 5개 → 인용 1개. URL 형태와 실제 주장 지지는 별도 검사 |
| 서비스/SSE | 14개 요청 정상 종료 | 느린 소비자·취소·빈 답변 결함은 모의 재현. 실제 장애 주입은 수행하지 않음 |

**결정:** LangGraph 라이브러리 자체 결함이라는 증거는 없다. 우선 애플리케이션의 입력 검증·정보 전달·종료 계약을 고치는 편이 작고 직접적인 변경이다. 새 노드·reranker·멀티 에이전트 추가는 아직 채택하지 않는다.

| 우선순위 | 제안할 최소 변경 | 승인 후 통과 기준 |
|---|---|---|
| 1 | 의도 JSON 타입·분류 허용값 검증 | Q02 오타가 불가능한 필터로 전달되지 않음; 정상 분류 유지; 잘못된 값 처리 정책 명시 |
| 2 | 이력/계약 정보 전달, 답변 이력 보존 | 실제 경계 회귀에서 전달·AI 역할 확인; 독립 질문 동작 유지 |
| 3 | 인용 근거 단위와 판본 보존 | 다른 조문·판본을 source_id만으로 합치지 않음; 동일 근거 중복은 제거 |
| 4 | 빈 응답·취소·느린 소비자 종료 | 빈 답변 성공 처리 방지, 소비자 종료 및 생산자 종료 계약 검증 |
| 후속 실험 | Q05 원문/요약, 필터 유무, Q09 판본 처리 | 같은 조건의 검색 결과를 사람 관련도·판본 검토와 함께 비교 |

서비스 변경은 아직 하지 않았다. 분류 오타를 바로잡기만 하면 정답이 나온다는 주장은 하지 않는다. 고정된 Before 점수도 변경하지 않는다.

### Critical Review와 완료 범위
- **누수:** 서비스에는 원문 질문만 전달했다. 정답 근거·평가 라벨을 넣지 않았다. 이번 질문들은 개발용이며 독립 holdout 결과가 아니다.
- **공정성:** 동일한 서비스·모델 pin·코퍼스를 확인했다. 계측 오버헤드와 모델 변동이 있어 Before/After 개선 수치로 보고하지 않는다.
- **정량/정성:** 관측 횟수·검색 개수·시간은 실측 요약이다. Q02/Q05/Q09 사례를 원인 검토에 사용했다. 새 답변의 사용자 점수·Recall/nDCG·법률적 정확성은 확정하지 않았다.
- **개선/회귀 사례:** 서비스를 바꾸지 않았으므로 개선·회귀 실험 사례는 해당 없음이다. 미해결 대표 사례는 Q02 분류 수정 후 품질, Q05 조건 누락, Q09 판본 적합성이다. 앞 절의 사용자 검토에서 지적된 나머지 질문도 보존한다.
- **복잡도:** 분석은 07 안에 유지하며 새 실행 프레임워크를 만들지 않았다. 실제 준비 중 발견한 저장 함수 import 누락만 03 원본 셀에서 수정했다.
- **안전성:** 전체 활성 경로 감사·기록 분석은 완료했다. 모든 분기의 실제 장애 검증이나 전체 검색 품질 평가가 완료된 것은 아니다. 서비스 수정과 Git 반영은 별도 승인 대상이다.

백업 SHA256: `4ded076ebafb1012987524e34ba329fb8a1e8902d259d43300369fdd36414171`. GPU는 백업 검증 후 중지했고 저장 공간은 보존했다.

In [24]:
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == value for p, value in before_hashes.items()), '[검증 실패] 원본 해시 보존·미측정 사용량: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(r["usage"] == "unknown" and r["cost"] == "unknown" for r in audit.values()), '[검증 실패] 원본 해시 보존·미측정 사용량: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
print('[검증][완료] Before·코퍼스·사용자 검토 보존 확인. 로컬 분석 종료, 추가 GPU 호출 없음.')

[검증][완료] Before·코퍼스·사용자 검토 보존 확인. 로컬 분석 종료, 추가 GPU 호출 없음.


## 부록 A. 실제 노드 기록 수집과 계측 검증
저장된 결과를 읽는 데는 이 부록을 실행할 필요가 없다. 수집 절차의 재현과 계측 검증을 위해 보존한다. 수집 함수는 입력·출력을 기록하고 실제 서비스의 반환값은 그대로 전달한다.

In [25]:
async def collect_node_audit(question):
    """id/question만 받아 실제 서비스의 노드 경계와 SSE 결과를 반환한다. 자동 저장/재시도는 없다."""
    import copy
    import time
    from contextlib import ExitStack
    from types import SimpleNamespace
    from unittest.mock import patch
    from harness.before import collect_answer
    import app.graph.pipeline as graph_module
    import app.services.chat_service as service
    import app.graph.nodes.generator as generator_module
    trace = {"nodes": [], "generation_messages": None}

    def snapshot(value):
        """메시지 객체를 JSON용 자료로 바꾸고 변경 전 값을 복사한다."""
        if hasattr(value, "model_dump"):
            return value.model_dump(mode="json")
        if isinstance(value, dict):
            return {k: snapshot(v) for k,v in value.items()}
        if isinstance(value, (list, tuple)):
            return [snapshot(v) for v in value]
        return copy.deepcopy(value)

    def observe(name, function):
        """노드 함수를 감싸 입력·출력·지연을 기록하고 반환값은 그대로 전달한다."""
        def wrapped(state):
            event = {"node":name, "input":snapshot(state), "output":None, "error":None}
            trace["nodes"].append(event)
            started = time.perf_counter()
            try:
                result = function(state)
                event["output"] = snapshot(result)
                return result
            except Exception as exc:
                event["error"] = type(exc).__name__
                raise
            finally:
                event["elapsed_ms"] = (time.perf_counter()-started)*1000
        return wrapped

    # 빌더를 한 번만 실행해 실제 생성 호출에 쓰인 메시지를 관찰한다.
    original_builder = generator_module._build_messages
    def observe_messages(state):
        """실제 메시지 빌더 반환값을 기록하고 그대로 전달한다."""
        messages = original_builder(state)
        trace["generation_messages"] = snapshot(messages)
        return messages

    import app.graph.nodes.understand_query as intent_module
    original_invoke = intent_module.llm.invoke
    def observe_intent(messages):
        """의도 모델에 실제 전달한 메시지와 원 응답을 기록한다."""
        trace["intent_messages"] = snapshot(messages)
        response = original_invoke(messages)
        trace["intent_response"] = snapshot(response)
        return response

    original_stream = graph_module.stream_generator
    def observe_stream(state):
        """스트리밍 생성 입력과 청크를 기록하며 청크를 수정하지 않는다."""
        event = {"node":"generator_stream", "input":snapshot(state), "chunks":[], "error":None}
        trace["nodes"].append(event)
        started = time.perf_counter()
        try:
            for chunk in original_stream(state):
                event["chunks"].append(chunk)
                yield chunk
        except Exception as exc:
            event["error"] = type(exc).__name__
            raise
        finally:
            event["elapsed_ms"] = (time.perf_counter()-started)*1000

    names = ["check_sensitive_info", "understand_query", "retrieve", "check_relevance", "resolve_citations"]
    with ExitStack() as stack:
        stack.enter_context(patch.object(generator_module, "_build_messages", observe_messages))
        stack.enter_context(patch.object(intent_module, "llm", SimpleNamespace(invoke=observe_intent)))
        for name in names:
            stack.enter_context(patch.object(graph_module, name, observe(name, getattr(graph_module, name))))
        observed_graph = graph_module.build_graph(include_generator=False)
        stack.enter_context(patch.object(service, "run_preformat", lambda query: observed_graph.invoke({"user_input":query})))
        stack.enter_context(patch.object(graph_module, "stream_generator", observe_stream))
        result = await collect_answer({"id":question["id"], "question":question["question"]}, service.stream_chat, trace)
    result["run_kind"] = "node_audit_not_before"
    result["node_status"] = {name: ("observed" if any(e["node"]==name for e in trace["nodes"]) else "not_reached")
                             for name in [*names,"generator_stream"]}
    return result

### 계측 자체의 무변경 검증
고정 응답으로 계측 없는 서비스와 계측 있는 서비스를 각각 실행한다. 시간 필드를 제외한 답변/인용/상태가 같아야 한다. 이 검증은 실제 서버의 모든 실행 동일성을 보장하지 않지만, 계측이 모의 정상 경로를 바꾸지 않는지 확인한다.

In [26]:
store.search.return_value = [doc]
with patch.object(retrieval, "_get_store", return_value=store), patch.object(gen.llm, "invoke", return_value=AIMessage(content=parsed), side_effect=None), patch.object(gen.streaming_llm, "stream", side_effect=None, return_value=[AIMessage(content="고정 답변")]):
    from harness.before import collect_answer
    plain = await collect_answer({"id":"fixture", "question":"모의 질문"}, chat.stream_chat, {})
    observed = await collect_node_audit({"id":"fixture", "question":"모의 질문"})
assert all(plain[k] == observed[k] for k in ["status","answer","citations","error"]), '[검증 실패] 계측 전후 응답 일치·노드 기록·원본 보존: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert all(v == "observed" for v in observed["node_status"].values()), '[검증 실패] 계측 전후 응답 일치·노드 기록·원본 보존: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
assert len(observed["trace"]["nodes"]) == 6, '[검증 실패] 계측 전후 응답 일치·노드 기록·원본 보존: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
print('[Mock][검증 완료] 계측 전후 답변·인용·상태가 같고 6개 노드 경계가 기록됐습니다.')
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == value for p,value in before_hashes.items()), '[검증 실패] 계측 전후 응답 일치·노드 기록·원본 보존: 예상값 불일치. 바로 위 입력과 실제 결과를 확인하세요.'
print('[검증][완료] 기존 Before·검토 의견·코퍼스 해시가 유지됐습니다.')

[Mock][검증 완료] 계측 전후 답변·인용·상태가 같고 6개 노드 경계가 기록됐습니다.


[검증][완료] 기존 Before·검토 의견·코퍼스 해시가 유지됐습니다.


### 실제 재수집: 환경 준비 → 실행·저장 → 호출
아래 함수 정의만 실행하면 모델을 호출하지 않는다. 마지막 호출 셀의 스위치는 기본 `False`다.

실제 실행은 새 비용·시간 승인, 별도 중지 도우미, 색인 사본 준비 후 **원격 새 커널**에서 이 부록의 수집 함수·승인 검사·준비 함수·저장 함수·호출 셀만 실행한다. 본문의 Mock 검사를 실행한 커널은 사용할 수 없다. 2026-10-20 실행 승인은 만료됐다.

승인·시간·환경 조건은 `if`와 예외로 검사한다. `assert`처럼 Python 최적화 옵션에서 생략되지 않는다. 호출 스위치만 켜는 것은 사용자 승인이 아니다. 원격 준비 이후에도 실행 직전과 질문 사이에 시간 제한을 확인한다.

In [27]:
def check_run_permission():
    """환경의 승인·종료 시각을 확인하고 유효한 종료 시각을 반환한다."""
    import math
    import os
    import time

    if os.environ.get("AIGO_NODE_AUDIT_APPROVED") != "yes":
        raise PermissionError("[실행] 신규 실행 승인이 필요합니다. 이전 승인을 재사용하지 마세요.")
    try:
        deadline = float(os.environ["AIGO_NODE_AUDIT_STOP_AT"])
    except (KeyError, ValueError) as exc:
        raise ValueError("[실행] 승인된 종료 시각을 AIGO_NODE_AUDIT_STOP_AT에 설정하세요.") from exc
    if not math.isfinite(deadline) or time.time() >= deadline:
        raise TimeoutError("[실행] 종료 시각이 유효하지 않거나 만료됐습니다. 새 승인을 확인하세요.")
    return deadline

In [28]:
def prepare_node_collection():
    """승인된 환경·코드·색인 사본을 확인하고 실행 설정을 반환한다."""
    check_run_permission()
    import os, sys, json, hashlib, subprocess, time
    from pathlib import Path
    from importlib.metadata import version
    from huggingface_hub import snapshot_download
    if any((name.startswith('app.') for name in sys.modules)):
        raise ValueError('[실행] app 모듈이 이미 로드됐습니다. 모의 검사와 분리한 새 커널을 사용하세요.')
    manifest = json.loads(Path("notebooks/redesign/data/before_manifest.json").read_text(encoding="utf-8"))
    cache = Path(os.environ["AIGO_CACHE_DIR"]) / "before" / manifest["corpus_sha256"]
    previous = cache / "runs" / "20261018T073552164357Z"
    old_runtime = json.loads((previous / "runtime.json").read_text(encoding="utf-8"))["runtime"]
    if hashlib.sha256((cache / 'corpus.jsonl').read_bytes()).hexdigest() != manifest['corpus_sha256']:
        raise ValueError('[실행] 코퍼스 해시가 다릅니다. 고정된 백업을 복구하세요.')
    if hashlib.sha256(Path('notebooks/redesign/eval/eval_set.json').read_bytes()).hexdigest() != manifest['eval_sha256']:
        raise ValueError('[실행] 평가셋 해시가 다릅니다. 승인된 질문 파일을 확인하세요.')
    files = subprocess.check_output(["git", "ls-files", "app"], text=True).splitlines()
    for name in files:
        baseline = subprocess.check_output(["git", "show", manifest["before_commit"] + ":" + name])
        if Path(name).read_bytes().replace(b'\r\n', b'\n') != baseline.replace(b'\r\n', b'\n'):
            raise ValueError(f"[실행] Before와 서비스 코드가 다릅니다: {name}")
    if not all((version(k) == v for k, v in old_runtime['execution_packages'].items())):
        raise ValueError('[실행] 패키지 버전이 Before와 다릅니다. 고정 환경을 복구하세요.')
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    for key, value in manifest["settings"].items():
        os.environ[key] = str(value)
    embedding_path = snapshot_download("nlpai-lab/KURE-v1", revision=old_runtime["embedding_revision"], allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    os.environ["EMBEDDING_MODEL"] = embedding_path
    os.environ["QDRANT_URL"] = ""
    os.environ["QDRANT_API_KEY"] = ""
    os.environ["QDRANT_PATH"] = os.environ["AIGO_AUDIT_INDEX"]
    os.environ["COLLECTION_LEGAL"] = "before_" + manifest["corpus_sha256"][:12]
    if not Path(os.environ['QDRANT_PATH']).is_dir():
        raise ValueError('[실행] 색인 사본 경로가 없습니다. AIGO_AUDIT_INDEX를 확인하세요.')
    if Path(os.environ['QDRANT_PATH']).resolve() == (previous / 'qdrant').resolve():
        raise ValueError('[실행] 원본 색인 경로입니다. 별도 사본을 지정하세요.')
    from app.graph.nodes.retrieve import _get_store
    store = _get_store()
    count = store.client.count(os.environ["COLLECTION_LEGAL"], exact=True).count
    if count != 2494:
        raise ValueError('[실행] 색인 문서 수가 2494개가 아닙니다. 색인 사본을 확인하세요.')
    audit_runtime = {"run_kind":"node_audit_not_before", "before_run":"20261018T073552164357Z",
        "generation_revision":old_runtime["generation_revision"], "embedding_revision":old_runtime["embedding_revision"],
        "execution_packages":{k:version(k) for k in old_runtime["execution_packages"]},
        "settings":manifest["settings"], "corpus_sha256":manifest["corpus_sha256"],
        "eval_sha256":manifest["eval_sha256"], "stop_at":float(os.environ["AIGO_NODE_AUDIT_STOP_AT"]),
        "approved_gpu_budget_usd":float(os.environ["AIGO_NODE_AUDIT_GPU_BUDGET"]), "point_count":count,
        "limitations":["모델 서버 실행 인자는 API/프로세스 사전 확인에 근거", "기존 Before 덮어쓰기 아님", "계측 지연 포함"]}
    print('[실행][준비 완료] 서비스·코퍼스·평가셋·패키지 일치 확인. 색인 사본의 문서 수: 2494개.')
    return audit_runtime

In [29]:
async def collect_and_save_nodes(audit_runtime):
    """승인된 14개 질문을 실행하고 질문별 기록을 새 디렉터리에 저장한다."""
    deadline = check_run_permission()
    import time
    import os
    import json
    from pathlib import Path
    from unittest.mock import Mock
    from harness.before import save_trace
    import app.llm as actual_llm
    if isinstance(actual_llm.llm, Mock):
        raise ValueError('[실행] Mock 모델이 로드됐습니다. 실제 수집용 새 커널을 사용하세요.')
    eval_file = Path("notebooks/redesign/eval/eval_set.json")
    questions = json.loads(eval_file.read_text(encoding="utf-8"))["items"]
    if len(questions) != 14:
        raise ValueError('[실행] 질문은 14개여야 합니다. 승인된 평가셋을 확인하세요.')
    output = Path(os.environ["AIGO_NODE_AUDIT_OUTPUT"])
    output.mkdir(parents=True, exist_ok=False)  # 기존 실행을 덮어쓰지 않는다.
    save_trace(output / "runtime.json", audit_runtime)
    for question in questions:
        if time.time() >= deadline:
            raise TimeoutError('[실행][시간 초과] 승인된 종료 시각에 도달했습니다. 저장된 결과를 보존하고 다음 질문은 실행하지 않습니다.')
        # 라벨·정답을 넘기지 않는다. 질문 하나가 끝날 때마다 비공개 경로에 저장한다.
        record = await collect_node_audit({"id":question["id"], "question":question["question"]})
        save_trace(output / (question["id"] + ".json"), record)
        print(f"[실행][저장 완료] {question['id']}: status={record['status']}; 질문별 원시 결과를 저장했습니다.")

### 승인·시간 경계의 로컬 검증
가짜 환경값과 고정 시간을 사용한다. 승인 누락·종료 시각 누락·만료·잘못된 숫자를 거부하고 유효한 시각만 반환해야 한다. 실제 승인이나 GPU 실행이 아니며 환경값은 검사 뒤 복원한다.

In [30]:
import os
from unittest.mock import patch

permission_cases = [
    ({}, PermissionError),
    ({"AIGO_NODE_AUDIT_APPROVED": "yes"}, ValueError),
    ({"AIGO_NODE_AUDIT_APPROVED": "yes", "AIGO_NODE_AUDIT_STOP_AT": "99"}, TimeoutError),
    ({"AIGO_NODE_AUDIT_APPROVED": "yes", "AIGO_NODE_AUDIT_STOP_AT": "nan"}, TimeoutError),
    ({"AIGO_NODE_AUDIT_APPROVED": "yes", "AIGO_NODE_AUDIT_STOP_AT": "invalid"}, ValueError),
    ({"AIGO_NODE_AUDIT_APPROVED": "yes", "AIGO_NODE_AUDIT_STOP_AT": "101"}, None),
]
for environment, expected_error in permission_cases:
    with patch.dict(os.environ, environment, clear=True), patch("time.time", return_value=100):
        try:
            deadline = check_run_permission()
        except (PermissionError, ValueError, TimeoutError) as exc:
            if type(exc) is not expected_error:
                raise AssertionError("승인/시간 검사에서 예상과 다른 예외가 발생했습니다.") from exc
        else:
            if expected_error is not None or deadline != 101:
                raise AssertionError("승인/시간 검사가 잘못된 실행을 허용했습니다.")
print("[Mock][검증 완료] 승인·시간 경계 6개 사례 확인. 실제 실행 승인은 생성하지 않았습니다.")

[Mock][검증 완료] 승인·시간 경계 6개 사례 확인. 실제 실행 승인은 생성하지 않았습니다.


In [31]:
RUN_LIVE_NODE_COLLECTION = False  # 신규 승인·환경 확인 후 이 한 곳에서만 변경한다.
print("[실행] 실제 노드 수집 요청:", RUN_LIVE_NODE_COLLECTION)
if RUN_LIVE_NODE_COLLECTION:
    runtime = prepare_node_collection()
    await collect_and_save_nodes(runtime)

[실행] 실제 노드 수집 요청: False
